In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'/Users/emran/items/medibot'

In [3]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [4]:
# Extract text from PDF files
def load_pdf_files(path):
    loader = DirectoryLoader(path, glob="**/*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents


In [5]:
extracted_data = load_pdf_files("data")

In [6]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_data(documents: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects that only contain the page_content and the source metadata.
    """
    minimal_data = []
    for doc in documents:
        minimal_doc = Document(
            page_content=doc.page_content,
            metadata={"source": doc.metadata.get("source", "")}
        )
        minimal_data.append(minimal_doc)
    return minimal_data

In [7]:
minimal_docs = filter_to_minimal_data(extracted_data)

In [8]:
# Split the document into smaller chunks
def text_splitter(documents: List[Document]) -> List[Document]:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    return text_splitter.split_documents(documents)

In [9]:
text_chunks = text_splitter(minimal_docs)
print(f"Number of text chunks: {len(text_chunks)}")

Number of text chunks: 5860


In [10]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings(model_name: str = "sentence-transformers/all-MiniLM-L6-v2") -> HuggingFaceEmbeddings:
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

In [11]:
embeddings = download_embeddings()

/var/folders/1t/v9yfyjdj6ls1n7h_f8l8wnfw0000gq/T/ipykernel_26317/2164781630.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)


In [12]:
vector = embeddings.embed_query("What is the purpose of this clinical trial?")
print(vector[:5])

[-0.02433726377785206, 0.08512768149375916, -0.007169554941356182, 0.004625589121133089, 0.056158117949962616]


In [13]:
print(f"Vector length: {len(vector)}")

Vector length: 384


In [14]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [15]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [16]:
from pinecone import Pinecone

pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_api_key)

In [17]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        vector_type="dense",
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
        deletion_protection="disabled",
        tags={
            "environment": "development"
        }
    )

index = pc.Index(index_name)

In [18]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(text_chunks, embeddings, index_name=index_name)

In [19]:
new_doc = Document(
    page_content="This is a new document to be added to the Pinecone index.",
    metadata={"source": "new_doc.pdf"}
)

docsearch.add_documents([new_doc])

['465a8097-d8d8-4c96-80e5-7b7fa0348df1']

In [20]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [21]:
retrieved_docs = retriever.invoke("What is the purpose of this clinical trial?")
retrieved_docs

[Document(id='32ad1ec3-c1c4-407d-9721-6d8554ca3cf7', metadata={'source': 'data/Medical_book.pdf'}, page_content='method of care with a new method, or the trials may be try-\ning to establish whether one treatment is more beneficial\nfor certain patients than others. Sometimes, a new treat-\nment that is not being offered on a wide scale may be avail-\nable to patients participating in clinical trials, but participat-\ning in the trials may involve some risk. To learn more about\nclinical trials, patients can call the National Cancer Institute\n(NCI) at 1-800-4-CANCER or visit the NCI web site for'),
 Document(id='e4b7122c-938f-4be9-8a04-04ad24493f73', metadata={'source': 'data/Medical_book.pdf'}, page_content='Purpose\nThe drugs described here are used to treat or prevent\nosteoporosis (brittle bone disease) in women past\nmenopause as well as older men. They also are used pre-\nscribed for Paget’s disease, a painful condition that weak-\nens and deforms bones, and they are used to con

In [22]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model="gpt-4o")

In [23]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [24]:
system_prompt = (
    "You are a helpful Medical assistant that provides information about clinical trials based on the retrieved documents."
    "Use the following retrieved documents to answer the question. If you don't know the answer, say you don't know."
    "Use three sentences at most to answer the question."
    "and keep the answer concise."
    "\n\n"
    "{context}"
    )

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])


In [25]:
question_answering_chain = create_stuff_documents_chain(
    llm=chatModel,
    prompt=prompt,
)

rag_chain = create_retrieval_chain(
    retriever,
    question_answering_chain,
)

In [26]:
response = rag_chain.invoke({"input": "What is Acromegaly and gigantism ?"})
print(response["answer"])

Acromegaly is a disorder caused by the abnormal release of a chemical from the pituitary gland, leading to increased growth in bone and soft tissue, along with various other bodily disturbances. If this abnormality occurs before bone growth stops, it results in gigantism, characterized by unusual height. When it occurs after bone growth stops, it is known as acromegaly.


In [27]:
response = rag_chain.invoke({"input": "What is the treatment for acne ?"})
print(response["answer"])

Treatment for acne depends on its severity. Mild noninflammatory acne is often treated with topical agents like tretinoin, benzoyl peroxide, adapalene, or salicylic acid. In cases of inflammation, topical antibiotics may be added, and improvement is typically seen within two to four weeks.
